In [1]:
"""
Pipeline Training Penuh
Big Data Challenge Satria Data 2026
Script ini menjalankan training penuh mulai dari Stage 1 (backbone
dibekukan) sampai Stage 2 (backbone dibuka penuh), hingga menghasilkan
satu file model terbaik berdasarkan Macro F1 Score pada data validasi.
Komponen arsitektur, loss, optimizer, dataset, dan EarlyStopping diimpor
dari file model_architecture_lengkap.py, jadi pastikan kedua file berada
pada folder yang sama.
Penyesuaian khusus untuk GPU dengan VRAM terbatas (RTX 3050, 4 GB) sudah
diterapkan di sini, yaitu batch size kecil dikombinasikan dengan gradient
accumulation, mixed precision training, dan gradient clipping.
"""
import os
import sys
# Wajib: cegah CUDA memory fragmentation crash (ExitCode 3221225477) di Windows
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.append(os.path.abspath("../../Model_Architecture"))
import time
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, classification_report
from model_architecture_lengkap import (
    SampahClassifier,
    hitung_class_counts_dari_manifest,
    create_loss_function,
    setup_optimizer_scheduler_stage1,
    setup_optimizer_scheduler_stage2,
    buat_dataloader,
    EarlyStopping,
    muat_bobot_terbaik,
    DEVICE,
    LABEL_MAP,
    TRAIN_MANIFEST_PATH,
    VAL_MANIFEST_PATH,
    KOLOM_LABEL,
    BEST_MODEL_STAGE1_PATH,
    BEST_MODEL_STAGE2_PATH,
    EPOCHS_STAGE1,
    EPOCHS_STAGE2,
    PATIENCE_EARLY_STOPPING,
    GRADIENT_CLIP_NORM,
)
# Override path manifest agar mengarah ke folder yang benar
TRAIN_MANIFEST_PATH = os.path.abspath("../../Preprocessing_Data/train_manifest.csv")
VAL_MANIFEST_PATH = os.path.abspath("../../Preprocessing_Data/val_manifest.csv")
print(f"Setup selesai. Device: {DEVICE}")
print(f"Train Manifest path: {TRAIN_MANIFEST_PATH}")
print(f"Val Manifest path: {VAL_MANIFEST_PATH}")
print(f"Train Manifest exists: {os.path.exists(TRAIN_MANIFEST_PATH)}")
print(f"Val Manifest exists: {os.path.exists(VAL_MANIFEST_PATH)}")


c:\Users\bagas\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Menggunakan device cuda
Setup selesai. Device: cuda
Train Manifest path: d:\Bagas\Satria_Data_BDC\BDC 2026\Preprocessing_Data\train_manifest.csv
Val Manifest path: d:\Bagas\Satria_Data_BDC\BDC 2026\Preprocessing_Data\val_manifest.csv
Train Manifest exists: True
Val Manifest exists: True


In [2]:
# ============================================================
# Konfigurasi khusus untuk GPU VRAM terbatas
# ============================================================
BATCH_SIZE_AKTUAL = 8
GRADIENT_ACCUMULATION_STEPS = 4
NUM_WORKERS = 0
# Path simpan model final di folder yang sama dengan notebook ini
NOTEBOOK_DIR = os.path.abspath("./")
BEST_MODEL_FINAL_PATH = os.path.join(NOTEBOOK_DIR, "best_model_final.pth")
print(f"Batch size aktual       : {BATCH_SIZE_AKTUAL}")
print(f"Gradient accumulation   : {GRADIENT_ACCUMULATION_STEPS} steps")
print(f"Efektif setara batch    : {BATCH_SIZE_AKTUAL * GRADIENT_ACCUMULATION_STEPS}")
print(f"Model final disimpan di : {BEST_MODEL_FINAL_PATH}")


Batch size aktual       : 8
Gradient accumulation   : 4 steps
Efektif setara batch    : 32
Model final disimpan di : d:\Bagas\Satria_Data_BDC\BDC 2026\Train_model\Best_model\best_model_final.pth


In [3]:
# ============================================================
# Fungsi Training Satu Epoch dengan Gradient Accumulation
# ============================================================
def train_one_epoch(model, loader, criterion, optimizer, scaler,
                     gradient_clip_norm=GRADIENT_CLIP_NORM,
                     accumulation_steps=GRADIENT_ACCUMULATION_STEPS):
    """
    Menjalankan satu epoch training memakai mixed precision dan gradient
    accumulation. Dengan batch size aktual kecil (misalnya 8) dan
    accumulation_steps 4, efeknya setara dengan batch size 32 dari sisi
    stabilitas gradien, tanpa membutuhkan VRAM sebesar batch size 32.
    """
    model.train()
    total_loss = 0.0
    jumlah_batch = 0
    use_amp = (DEVICE.type == "cuda")
    optimizer.zero_grad()
    for idx_batch, (gambar, label) in enumerate(loader):
        gambar = gambar.to(DEVICE, non_blocking=True)
        label = label.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(DEVICE.type, enabled=use_amp):
            output = model(gambar)
            loss = criterion(output, label)
            loss_dibagi = loss / accumulation_steps
        scaler.scale(loss_dibagi).backward()
        langkah_terakhir = (idx_batch + 1) == len(loader)
        if (idx_batch + 1) % accumulation_steps == 0 or langkah_terakhir:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        total_loss += loss.item()
        jumlah_batch += 1
        # Progress setiap 100 batch
        if (idx_batch + 1) % 100 == 0 or langkah_terakhir:
            vram_mb = torch.cuda.memory_allocated(0) / 1024**2 if DEVICE.type == "cuda" else 0
            print(f"  [{idx_batch+1:4d}/{len(loader)}] loss={loss.item():.4f}  VRAM={vram_mb:.0f}MB",
                  flush=True)
    rata_loss = total_loss / jumlah_batch
    return rata_loss
# ============================================================
# Fungsi Validasi Satu Epoch
# ============================================================
def validasi_satu_epoch(model, loader, criterion):
    """
    Menjalankan validasi satu epoch, mengembalikan rata rata loss serta
    Macro F1 Score yang menjadi metrik utama kompetisi ini.
    """
    model.eval()
    total_loss = 0.0
    jumlah_batch = 0
    use_amp = (DEVICE.type == "cuda")
    seluruh_prediksi = []
    seluruh_label = []
    with torch.no_grad():
        for gambar, label in loader:
            gambar = gambar.to(DEVICE, non_blocking=True)
            label = label.to(DEVICE, non_blocking=True)
            with torch.amp.autocast(DEVICE.type, enabled=use_amp):
                output = model(gambar)
                loss = criterion(output, label)
            total_loss += loss.item()
            jumlah_batch += 1
            prediksi = torch.argmax(output, dim=1)
            seluruh_prediksi.extend(prediksi.cpu().numpy().tolist())
            seluruh_label.extend(label.cpu().numpy().tolist())
    rata_loss = total_loss / jumlah_batch
    macro_f1 = f1_score(seluruh_label, seluruh_prediksi, average="macro")
    return rata_loss, macro_f1, seluruh_label, seluruh_prediksi
print("Fungsi train_one_epoch dan validasi_satu_epoch siap.")


Fungsi train_one_epoch dan validasi_satu_epoch siap.


In [ ]:
# ============================================================
# Alur Stage 1
# ============================================================
def jalankan_stage1(model, loader_train, loader_val, class_counts):
    print("\nMemulai Stage 1 — backbone dibekukan, hanya melatih head")
    model.freeze_backbone()
    criterion = create_loss_function(class_counts)
    optimizer, scheduler = setup_optimizer_scheduler_stage1(model)
    use_amp = (DEVICE.type == "cuda")
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=use_amp)
    early_stopping = EarlyStopping(patience=PATIENCE_EARLY_STOPPING, mode="max")
    for epoch in range(1, EPOCHS_STAGE1 + 1):
        waktu_mulai = time.time()
        print(f"\n--- Stage 1 Epoch {epoch}/{EPOCHS_STAGE1} ---")
        rata_loss_train = train_one_epoch(model, loader_train, criterion, optimizer, scaler)
        rata_loss_val, macro_f1_val, _, _ = validasi_satu_epoch(model, loader_val, criterion)
        scheduler.step()
        durasi = time.time() - waktu_mulai
        print(
            f"HASIL  loss_train={rata_loss_train:.4f}  loss_val={rata_loss_val:.4f}  "
            f"macro_f1={macro_f1_val:.4f}  waktu={durasi:.1f}s"
        )
        early_stopping(macro_f1_val, model, BEST_MODEL_STAGE1_PATH)
        if early_stopping.early_stop:
            print("Stage 1 dihentikan lebih awal")
            break
    model = muat_bobot_terbaik(model, BEST_MODEL_STAGE1_PATH)
    return model
# ============================================================
# Alur Stage 2
# ============================================================
def jalankan_stage2(model, loader_train, loader_val, class_counts):
    print("\nMemulai Stage 2 — backbone dibuka penuh")
    model.unfreeze_backbone()
    criterion = create_loss_function(class_counts)
    optimizer, scheduler = setup_optimizer_scheduler_stage2(model)
    use_amp = (DEVICE.type == "cuda")
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=use_amp)
    early_stopping = EarlyStopping(patience=PATIENCE_EARLY_STOPPING, mode="max")
    for epoch in range(1, EPOCHS_STAGE2 + 1):
        waktu_mulai = time.time()
        print(f"\n--- Stage 2 Epoch {epoch}/{EPOCHS_STAGE2} ---")
        rata_loss_train = train_one_epoch(model, loader_train, criterion, optimizer, scaler)
        rata_loss_val, macro_f1_val, _, _ = validasi_satu_epoch(model, loader_val, criterion)
        scheduler.step()
        durasi = time.time() - waktu_mulai
        print(
            f"HASIL  loss_train={rata_loss_train:.4f}  loss_val={rata_loss_val:.4f}  "
            f"macro_f1={macro_f1_val:.4f}  waktu={durasi:.1f}s"
        )
        early_stopping(macro_f1_val, model, BEST_MODEL_STAGE2_PATH)
        if early_stopping.early_stop:
            print("Stage 2 dihentikan lebih awal")
            break
    model = muat_bobot_terbaik(model, BEST_MODEL_STAGE2_PATH)
    return model, criterion
print("Fungsi jalankan_stage1 dan jalankan_stage2 siap.")


Fungsi jalankan_stage1 dan jalankan_stage2 siap.


: 

In [ ]:
# ============================================================
# Alur Utama — Jalankan cell ini untuk memulai training
# ============================================================
torch.cuda.empty_cache()  # Bersihkan VRAM sebelum mulai
print(f"Menggunakan device      : {DEVICE}")
print(f"Batch size aktual       : {BATCH_SIZE_AKTUAL}")
print(f"Gradient accumulation   : {GRADIENT_ACCUMULATION_STEPS} steps")
print(f"Efektif setara batch    : {BATCH_SIZE_AKTUAL * GRADIENT_ACCUMULATION_STEPS}")
print()
class_counts = hitung_class_counts_dari_manifest(TRAIN_MANIFEST_PATH, KOLOM_LABEL, LABEL_MAP)
print("\nMemuat DataLoader...")
loader_train, loader_val = buat_dataloader(
    TRAIN_MANIFEST_PATH, VAL_MANIFEST_PATH, LABEL_MAP,
    batch_size=BATCH_SIZE_AKTUAL,
    num_workers=NUM_WORKERS,
)
print("\nMemuat model EfficientNetV2-S (pretrained)...")
model = SampahClassifier(num_classes=3, pretrained=True)
model.to(DEVICE)
vram_mb = torch.cuda.memory_allocated(0) / 1024**2
print(f"Model siap. VRAM terpakai: {vram_mb:.1f} MB")
print()
# ---- Stage 1 ----
model = jalankan_stage1(model, loader_train, loader_val, class_counts)
# ---- Stage 2 ----
model, criterion_terakhir = jalankan_stage2(model, loader_train, loader_val, class_counts)
# ---- Simpan model final ----
torch.save(model.state_dict(), BEST_MODEL_FINAL_PATH)
print(f"\nModel terbaik akhir disimpan pada {BEST_MODEL_FINAL_PATH}")
# ---- Evaluasi akhir ----
_, macro_f1_final, label_final, prediksi_final = validasi_satu_epoch(model, loader_val, criterion_terakhir)
nama_kelas = sorted(LABEL_MAP, key=lambda x: LABEL_MAP[x])
print(f"\nMacro F1 Score akhir: {macro_f1_final:.4f}")
print("\nClassification Report:")
print(classification_report(label_final, prediksi_final, target_names=nama_kelas))
print("\nTraining selesai!")
print(f"  Stage 1 terbaik : {BEST_MODEL_STAGE1_PATH}")
print(f"  Stage 2 terbaik : {BEST_MODEL_STAGE2_PATH}")
print(f"  Model final     : {BEST_MODEL_FINAL_PATH}")


In [ ]:
import os
import os
import sys
# Wajib: cegah CUDA memory fragmentation crash (ExitCode 3221225477) di Windows
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.append(os.path.abspath("../../Model_Architecture"))
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
from torchvision import transforms
from PIL import Image
# Import komponen dari arsitektur
from model_architecture_lengkap import (
    KOLOM_PATH, KOLOM_LABEL, LABEL_MAP, MANIFEST_PATH, 
    buat_dataloader
)
# Sesuaikan path manifest jika dijalankan dari folder ini
TRAIN_MANIFEST_PATH = os.path.abspath("../../Preprocessing_Data/train_manifest.csv")
VAL_MANIFEST_PATH = os.path.abspath("../../Preprocessing_Data/val_manifest.csv")
def visualize_dataframe(manifest_path, samples_per_class=4):
    """
    Fungsi untuk visualisasi langsung dari DataFrame / CSV
    sebelum masuk ke DataLoader.
    """
    print(f"Membaca DataFrame dari {manifest_path}...")
    df = pd.read_csv(manifest_path)
    labels = df[KOLOM_LABEL].unique()
    
    fig, axes = plt.subplots(len(labels), samples_per_class, figsize=(samples_per_class * 4, len(labels) * 4))
    fig.suptitle("Visualisasi Langsung dari DataFrame (Original Images)", fontsize=16, y=1.02)
    
    for i, label in enumerate(sorted(labels)):
        subset = df[df[KOLOM_LABEL] == label]
        # Ambil sampel acak
        if len(subset) >= samples_per_class:
            subset = subset.sample(samples_per_class)
        else:
            subset = subset.sample(samples_per_class, replace=True)
            
        for j, (_, row) in enumerate(subset.iterrows()):
            ax = axes[i, j]
            try:
                img_path = row[KOLOM_PATH]
                img = Image.open(img_path)
                ax.imshow(img)
                ax.set_title(f"Label Asli: {label}", fontsize=12)
                ax.axis('off')
            except Exception as e:
                ax.set_title("Error loading image")
                ax.axis('off')
                
    plt.tight_layout()
    plt.show()
def visualize_dataloader(train_manifest_path, val_manifest_path, label_map, num_images=12):
    """
    Fungsi untuk visualisasi output dari DataLoader
    setelah melalui transformasi/augmentasi (tensor).
    """
    print("\nMembuat DataLoader untuk visualisasi...")
    loader_train, _ = buat_dataloader(
        TRAIN_MANIFEST_PATH, VAL_MANIFEST_PATH, label_map,
        batch_size=num_images, 
        num_workers=0
    )
    
    # Ambil 1 batch dari loader_train
    images, labels = next(iter(loader_train))
    
    # Inversi map (index to label string)
    idx_to_label = {v: k for k, v in label_map.items()}
    
    # Denormalisasi untuk visualisasi matplotlib
    # std=[0.229, 0.224, 0.225], mean=[0.485, 0.456, 0.406]
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )
    
    cols = 4
    rows = int(np.ceil(num_images / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    fig.suptitle("Visualisasi Setelah DataLoader & Augmentasi (Tensor)", fontsize=16, y=1.02)
    
    axes = axes.flatten()
    for i in range(num_images):
        if i < len(images):
            img_tensor = images[i]
            label_idx = labels[i].item()
            label_str = idx_to_label[label_idx]
            
            # Denormalize dan ubah CHW -> HWC
            img_tensor = inv_normalize(img_tensor)
            img_np = img_tensor.permute(1, 2, 0).numpy()
            img_np = np.clip(img_np, 0, 1) # Pastikan rentang 0-1
            
            axes[i].imshow(img_np)
            axes[i].set_title(f"Target: {label_str} ({label_idx})", fontsize=12)
            axes[i].axis('off')
        else:
            axes[i].axis('off')
            
    plt.tight_layout()
    plt.show()
if __name__ == "__main__":
    # 1. Visualisasi DataFrame (gambar asli)
    visualize_dataframe(MANIFEST_PATH, samples_per_class=4)
    
    # 2. Visualisasi DataLoader (setelah augmentasi dan pemetaan label index)
    visualize_dataloader(TRAIN_MANIFEST_PATH, VAL_MANIFEST_PATH, LABEL_MAP, num_images=12)
